In [37]:
import pandas as pd 
import os 
import logging 
import traceback
import re
from pathlib import Path
from basicprocess import create_folder, outputlog, findfiles, read_combined_dataframe
from TDXdataframe import read_businfo_xml
from openpyxl import load_workbook


In [3]:
def get_taipeibusreport(odspath):

    logging.info("開始讀取台北市營運月報")
    logging.info(f"台北市公車營運月報的原始檔案為{odspath}")

    sheet_names = pd.ExcelFile(odspath).sheet_names
    dfs = []
    for sheet in sheet_names:
        odsdf = pd.read_excel(odspath, sheet_name=sheet, engine='odf')
        odsdf['Time'] = sheet
        dfs.append(odsdf)

    df = pd.concat(dfs, ignore_index=True)

    logging.info("台北市營運月報整併完成")
    

    df['年'] = df['Time'].str[:3].astype(int) + 1911
    df['月'] = df['Time'].str[3:].astype(int)
    # df['資料時間'] = pd.to_datetime(df['年'].astype(str) + '-' + df['月'].astype(str)).dt.to_period('M')
    df['資料時間'] = pd.to_datetime(
        df['年'].astype(str) + '-' + df['月'].astype(str).str.zfill(2),
        format='%Y-%m',
        errors='raise').dt.to_period('M')

    cols = ['資料時間'] + [c for c in df.columns if c not in ['年', '月', 'Time', '資料時間']]
    df = df.reindex(columns=cols)

    logging.info("輸出指定格式")

    rename_dist = {'資料時間':'Month',
                '客運業者':'OperatorName', 
                '路線代碼':'RouteID', 
                '路線別':'RouteName', 
                '總班次':'Shifts', 
                '總載客人次':'Passenger',
                '總行駛里程':'Miles',
                '延人公里':'PaxKm', 
                '總營收': 'Revenue'}
    df = df[list(rename_dist)]
    df = df.rename(columns = rename_dist)

    return df

def check_if_morethanone(df, checklists, timecolumn, warningfolder, dataname = '資料'):
    """
    檢查每個 timecolumn 內，checklists 是否有重複值

    Parameters
    ----------
    df : pandas.DataFrame
    checklists : list
        需要檢查是否重複的欄位
    timecolumn : str
        時間欄位（例如 Month）

    Returns
    -------
    duplicated_df : pandas.DataFrame
        含有重複資料的 dataframe（只保留重複者）
    summary : pandas.DataFrame
        每個月份重複筆數的摘要
    """

    # 找出在「同一個月 + checklists」下重複的資料
    mask = df.duplicated(subset=[timecolumn] + checklists, keep=False)
    duplicated_df = df[mask].sort_values([timecolumn] + checklists)

    if len(duplicated_df) > 0:
        logging.warning(f"{dataname} 有重複的資料")

        # 每月重複筆數摘要
        summary = (
            duplicated_df
            .groupby(timecolumn)
            .size()
            .reset_index(name='DuplicatedRows')
        )

        outputfile = os.path.join(warningfolder, f"{dataname}重複資料.xlsx")

        with pd.ExcelWriter(outputfile, engine='xlsxwriter') as writer:
            duplicated_df.to_excel(writer, index=True, sheet_name='有重複的資料')
            summary.to_excel(writer, index=True, sheet_name='每月重複筆數')

        logging.info(f"{dataname}路線營運月報，路徑：{outputfile}")

        return False     

    else:
        logging.info(f"{dataname}在{checklists}的組合底下沒有重複資料")
        return True

def get_businfo():
    businfos = []
    for xmlpath in findfiles(filefolderpath=os.path.join(os.getcwd(), '..', '00_TDX資料下載', '03公車路線營運資料'), filetype='xml'):
        businfo = read_businfo_xml(xml_path=xmlpath)
        businfos.append(businfo)
    businfo = pd.concat(businfos)

    return businfo

In [ ]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '04_營運月報整理.log'))
if os.path.exists(logfile):
    os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

def main():
    logging.info('Start Processing...')

    outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
    monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))
    finalorganizedfile = os.path.join(monthlyreport_organized_folder, '月報統計數量.xlsx')
    
    # 處理台北公車路線資料
    taipeidf = get_taipeibusreport(r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\04臺北\01公運處\附2-項目(二)臺北市營運資料(市區公車)v1.ods")
    taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
    logging.info('輸出臺北市公車營運月報整理結果')
    temp = check_if_morethanone(df = taipeidf, 
                                checklists = ['RouteID', 'RouteName' ], 
                                timecolumn = 'Month', 
                                dataname = '臺北市公車營運月報', 
                                warningfolder=create_folder(os.path.join(monthlyreport_organized_folder, 'error')))
    if temp == False:
        logging.info("臺北市公車因為有多個OpeartionName經營同一條路線")
        taipeidf = taipeidf.drop(columns='OperatorName').groupby(['Month','RouteName']).agg({'Shifts':'sum',
                                                                                             'Passenger':'sum',
                                                                                             'Miles':'sum',
                                                                                             'PaxKm':'sum',
                                                                                             'Revenue':'sum'}).reset_index()
        taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
        logging.info('重新輸出臺北市公車營運月報整理結果')

    del temp

    alldf = read_combined_dataframe(findfiles(monthlyreport_organized_folder, 'xlsx', recursive=False))
    alldf.to_excel(finalorganizedfile)


    logging.info('Finished Processing.')


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)

In [ ]:
outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))
taipeidf = read_combined_dataframe(findfiles(monthlyreport_organized_folder, 'xlsx', recursive=False), filepath=False)

# Trial 

In [35]:
def flatten_columns(multi_cols):
    flat_cols = []
    for col in multi_cols:
        parts = [
            str(v)
            for v in col
            if v is not None and not pd.isna(v) and str(v).strip() != ""
        ]
        flat_cols.append("_".join(parts))
    return flat_cols

def simplify_col(c: str) -> str:
    # 1. 移除單位 (括號內)
    c = re.sub(r"_?\([^)]*\)", "", c)

    # 2. 類別縮寫
    c = c.replace("路線營運資料_", "路線_")
    c = c.replace("車輛情形_", "車輛_")
    c = c.replace("包車出租_", "包車_")

    # 3. 常見冗字精簡
    c = c.replace("營業行車次數", "行車次數")
    c = c.replace("營業行駛里程", "行駛里程")
    c = c.replace("營業里程", "營業里程")
    c = c.replace("行駛延日車數", "延日車數")

    # 4. 多餘底線清理
    c = re.sub(r"__+", "_", c).strip("_")

    return c

def read_specific_data(excelfilepath, sheetname, cell):
    """
    讀取指定 Excel 檔案中，特定工作表與儲存格位置的資料。
    
    Parameters:
        excelfilepath (str): Excel 檔案的完整路徑
        sheetname (str): 工作表名稱
        cell (str): 儲存格位置，例如 'B5'
        
    Returns:
        value: 儲存格中的資料（任何類型）
    """
    wb = load_workbook(excelfilepath, data_only=True)
    ws = wb[sheetname]
    return ws[cell].value

def get_excel_sheet_names(path):
    """
    取得 Excel 檔案中的所有工作表名稱。

    Args:
        path (str): Excel 檔案的路徑。

    Returns:
        list: 工作表名稱列表。
    """
    try:
        sheet_names = pd.ExcelFile(path).sheet_names
        return sheet_names
    except FileNotFoundError:
        print(f"檔案不存在：{path}")
        return []
    except Exception as e:
        print(f"發生錯誤：{e}")
        return []

In [50]:
folder = r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\06桃園\02桃園市區客運業者營運概況"

def combined_all_bus_monthlyreport(folder):

    dfs = []
    files = findfiles(folder, 'xlsx')
    for file in files :
        sheetnames = get_excel_sheet_names(file)
        if '2522-02-01-2' in sheetnames:
            sheetname = '2522-02-01-2'
        else:
            sheetname = sheetnames[0]
        timetext = read_specific_data(file, sheetname, 'A4')
        m = re.search(r"(\d+)\s*年\s*(\d+)\s*月", timetext)
        dyear = int(m.group(1))
        dmonth = int(m.group(2))
        df = pd.read_excel(file, skiprows=4, header = [0, 1, 2, 3, 4])
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None if pd.isna(v) else str(v).replace("\n", "").replace("\u3000", "").strip()
                for v in col
            )
            for col in df.columns
        )
        df.columns = pd.MultiIndex.from_tuples(
            tuple(
                None
                if (
                    pd.isna(v)
                    or (isinstance(v, str) and v.startswith("Unnamed:"))
                )
                else v
                for v in col
            )
            for col in df.columns
        )
        df.columns = flatten_columns(df.columns)
        df.columns = [simplify_col(c) for c in df.columns]
        df['市區客運業者家數'] = pd.to_numeric(df['市區客運業者家數'], errors='coerce')
        df = df[(~df['市區客運業者家數'].isna()) & (df['項目'] != '總計')]
        df['年'] = dyear
        df['月'] = dmonth

        cols = df.columns.tolist()
        new_cols = ['年', '月'] + [c for c in cols if c not in ['年', '月']]
        df = df[new_cols]
        dfs.append(df)

    df = pd.concat(dfs)
    return df.sort_values(['年', '月']).reset_index()
combined_all_bus_monthlyreport(folder=folder)

,index,年,月,項目,市區客運業者家數,路線_核定路線數,路線_核定路線數_幸福巴士,路線_核定路線數_幸福小黃,路線_營業里程,路線_行車次數,...,車輛_營業車輛數_小客車,車輛_電動車輛數,車輛_無障礙車輛數,車輛_無障礙車輛數_低地板,車輛_延日車數,車輛_燃料消耗量_柴油,車輛_燃料消耗量_汽油,車輛_燃料消耗量_液化石油氣,包車_客運人數,包車_延日車數
0,1,113,1,亞通客運,1.0,31.0,0.0,0.0,511.60,15489,...,0.0,0.0,37.0,0.0,2089.0,97533.0,0.0,0,0.0,0.0
1,2,113,1,指南客運,1.0,4.0,0.0,0.0,109.60,4365,...,0.0,0.0,8.0,7.0,454.0,42360.0,0.0,0,0.0,0.0
2,3,113,1,統聯客運,1.0,12.0,0.0,0.0,283.05,13596,...,0.0,0.0,31.0,25.0,2759.0,132339.1,0.0,0,86.0,2.0
3,4,113,1,三重客運,1.0,6.0,0.0,0.0,85.98,6371,...,0.0,0.0,13.0,7.0,806.0,44659.29,0.0,0,0.0,0.0
4,5,113,1,中壢客運,1.0,17.0,0.0,0.0,329.50,13290,...,0.0,0.0,34.0,29.0,2449.0,81133.2,0.0,0,2082.0,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,4,114,4,三重客運,1.0,6,0,0,163.95,3564,...,0,0.0,9.0,3.0,690.0,23709.7,0,0,0,0.0
124,5,114,4,中壢客運,1.0,16,0,0,431.15,10261,...,0,0.0,32.0,27.0,2340.0,84423.3,0,0,1436,34.0
125,6,114,4,桃園客運,1.0,178,8,0,4044.63,63840,...,0,7.0,251.0,228.0,10582.0,642095.1,0,0,386034,2820.0
126,7,114,4,金台通運,1.0,4,0,0,42.02,2040,...,0,0.0,0.0,0.0,240.0,8687.44,0,0,19967,240.0


In [48]:
df


,年,月,項目,市區客運業者家數,路線_核定路線數,路線_核定路線數_幸福巴士,路線_核定路線數_幸福小黃,路線_營業里程,路線_行車次數,路線_行車次數_幸福巴士,...,車輛_營業車輛數_小客車,車輛_電動車輛數,車輛_無障礙車輛數,車輛_無障礙車輛數_低地板,車輛_延日車數,車輛_燃料消耗量_柴油,車輛_燃料消耗量_汽油,車輛_燃料消耗量_液化石油氣,包車_客運人數,包車_延日車數
1,114,4,亞通客運,1.0,35,0,0,932.83,16617,0,...,0,0.0,37.0,0.0,1770.0,213408.79,0,0,0,0.0
2,114,4,指南客運,1.0,4,0,0,154.55,2231,0,...,0,0.0,7.0,6.0,421.0,38552,0,0,0,0.0
3,114,4,統聯客運,1.0,13,0,0,462.95,10807,0,...,0,0.0,29.0,23.0,3180.0,135893,0,0,172,4.0
4,114,4,三重客運,1.0,6,0,0,163.95,3564,0,...,0,0.0,9.0,3.0,690.0,23709.7,0,0,0,0.0
5,114,4,中壢客運,1.0,16,0,0,431.15,10261,0,...,0,0.0,32.0,27.0,2340.0,84423.3,0,0,1436,34.0
6,114,4,桃園客運,1.0,178,8,0,4044.63,63840,580,...,0,7.0,251.0,228.0,10582.0,642095.1,0,0,386034,2820.0
7,114,4,金台通運,1.0,4,0,0,42.02,2040,0,...,0,0.0,0.0,0.0,240.0,8687.44,0,0,19967,240.0
8,114,4,台灣真好,1.0,22,22,0,892.60,513,513,...,22,0.0,0.0,0.0,302.0,1301.84,0,0,0,0.0


,項目,市區客運業者家數,路線_核定路線數,路線_核定路線數_幸福巴士,路線_核定路線數_幸福小黃,路線_營業里程,路線_行車次數,路線_行車次數_幸福巴士,路線_行車次數_幸福小黃,路線_行駛里程,...,車輛_營業車輛數_小客車,車輛_電動車輛數,車輛_無障礙車輛數,車輛_無障礙車輛數_低地板,車輛_延日車數,車輛_燃料消耗量_柴油,車輛_燃料消耗量_汽油,車輛_燃料消耗量_液化石油氣,包車_客運人數,包車_延日車數
1,亞通客運,1.0,35,0,0,932.83,16617,0,0,448158.45,...,0,0.0,37.0,0.0,1770.0,213408.79,0,0,0,0.0
2,指南客運,1.0,4,0,0,154.55,2231,0,0,106421.45,...,0,0.0,7.0,6.0,421.0,38552,0,0,0,0.0
3,統聯客運,1.0,13,0,0,462.95,10807,0,0,377365,...,0,0.0,29.0,23.0,3180.0,135893,0,0,172,4.0
4,三重客運,1.0,6,0,0,163.95,3564,0,0,84154.3,...,0,0.0,9.0,3.0,690.0,23709.7,0,0,0,0.0
5,中壢客運,1.0,16,0,0,431.15,10261,0,0,163109.6,...,0,0.0,32.0,27.0,2340.0,84423.3,0,0,1436,34.0
6,桃園客運,1.0,178,8,0,4044.63,63840,580,0,1248785.4,...,0,7.0,251.0,228.0,10582.0,642095.1,0,0,386034,2820.0
7,金台通運,1.0,4,0,0,42.02,2040,0,0,22098,...,0,0.0,0.0,0.0,240.0,8687.44,0,0,19967,240.0
8,台灣真好,1.0,22,22,0,892.60,513,513,0,16657.3,...,22,0.0,0.0,0.0,302.0,1301.84,0,0,0,0.0


In [30]:
df

,項目,市區客運業者家數,路線_核定路線數,路線_核定路線數_幸福巴士,路線_核定路線數_幸福小黃,路線_營業里程,路線_行車次數,路線_行車次數_幸福巴士,路線_行車次數_幸福小黃,路線_行駛里程,...,車輛_營業車輛數_小客車,車輛_電動車輛數,車輛_無障礙車輛數,車輛_無障礙車輛數_低地板,車輛_延日車數,車輛_燃料消耗量_柴油,車輛_燃料消耗量_汽油,車輛_燃料消耗量_液化石油氣,包車_客運人數,包車_延日車數
0,總計,8.0,278,30,0,7124.68,109873,1093,0,2466749.5,...,22,7.0,365.0,287.0,19525.0,1148071.17,0,0,407609,3098.0
1,亞通客運,1.0,35,0,0,932.83,16617,0,0,448158.45,...,0,0.0,37.0,0.0,1770.0,213408.79,0,0,0,0.0
2,指南客運,1.0,4,0,0,154.55,2231,0,0,106421.45,...,0,0.0,7.0,6.0,421.0,38552,0,0,0,0.0
3,統聯客運,1.0,13,0,0,462.95,10807,0,0,377365,...,0,0.0,29.0,23.0,3180.0,135893,0,0,172,4.0
4,三重客運,1.0,6,0,0,163.95,3564,0,0,84154.3,...,0,0.0,9.0,3.0,690.0,23709.7,0,0,0,0.0
5,中壢客運,1.0,16,0,0,431.15,10261,0,0,163109.6,...,0,0.0,32.0,27.0,2340.0,84423.3,0,0,1436,34.0
6,桃園客運,1.0,178,8,0,4044.63,63840,580,0,1248785.4,...,0,7.0,251.0,228.0,10582.0,642095.1,0,0,386034,2820.0
7,金台通運,1.0,4,0,0,42.02,2040,0,0,22098,...,0,0.0,0.0,0.0,240.0,8687.44,0,0,19967,240.0
8,台灣真好,1.0,22,22,0,892.60,513,513,0,16657.3,...,22,0.0,0.0,0.0,302.0,1301.84,0,0,0,0.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
